<a href="https://colab.research.google.com/github/AbhishekShah272002/Deep-Learning-and-NLP/blob/main/Transformer_Based_Household_Power_Consumption_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout, # Randomly drops some neuron outputs during training to help reduce overfitting.
    LayerNormalization, # Normalizes intermediate values inside the Transformer and helps make training more stable.
    MultiHeadAttention,  # This is the main Transformer component.It allows the model to examine the 60 previous time steps and learn relationships among them.
    GlobalAveragePooling1D # convert the sequence representation into a fixed-size representation before final prediction.
)

from tensorflow.keras.optimizers import Adam

In [5]:
df = pd.read_csv(
    "household_power_consumption.txt",
    sep=";"
)

df.head()

/tmp/ipykernel_6282/3746150651.py:1: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


In [6]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

print("\nDataset Information:")
df.info()

Dataset Shape: (81012, 9)

Column Names:
Index(['Date', 'Time', 'Global_active_power', 'Global_reactive_power',
       'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2',
       'Sub_metering_3'],
      dtype='object')

Data Types:
Date                      object
Time                      object
Global_active_power       object
Global_reactive_power     object
Voltage                   object
Global_intensity          object
Sub_metering_1            object
Sub_metering_2            object
Sub_metering_3           float64
dtype: object

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81012 entries, 0 to 81011
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Date                   81012 non-null  object 
 1   Time                   81012 non-null  object 
 2   Global_active_power    81012 non-null  object 
 3   Global_reactive_power  81012 non-null  object 


In [7]:
print("Missing Values:")
print(df.isnull().sum())

print("\nNumber of '?' values:")
print((df == '?').sum())

Missing Values:
Date                     0
Time                     0
Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           1
Sub_metering_3           7
dtype: int64

Number of '?' values:
Date                     0
Time                     0
Global_active_power      6
Global_reactive_power    6
Voltage                  6
Global_intensity         6
Sub_metering_1           6
Sub_metering_2           6
Sub_metering_3           0
dtype: int64


In [8]:
df.replace('?', np.nan, inplace=True)

df.dropna(inplace=True)

print("Missing Values After Cleaning:")
print(df.isnull().sum())

print("\nUpdated Dataset Shape:", df.shape)

Missing Values After Cleaning:
Date                     0
Time                     0
Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64

Updated Dataset Shape: (81005, 9)


In [9]:
df['DateTime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%d/%m/%Y %H:%M:%S'
)

df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0,2006-12-16 17:24:00
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0,2006-12-16 17:27:00
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0,2006-12-16 17:28:00


In [10]:
df['Global_active_power'] = pd.to_numeric(
    df['Global_active_power']
)

print(df['Global_active_power'].dtype)
print(df['Global_active_power'].head())

float64
0    4.216
1    5.360
2    5.374
3    5.388
4    3.666
Name: Global_active_power, dtype: float64
